# ALS gut microbiome reanalysis (PRJNA566436) — upgraded notebook

Reanalysis of Hertzberg et al. 2021: 9 ALS patients vs. their 9 spouse controls (16S V4, DADA2 ASVs, SILVA 138).

**Upgrades over v1:** depth diagnostics; richness/Pielou/rarefaction sensitivity; *exact* paired (within-household swap) PERMANOVA with 2^9 = 512 permutations; PERMDISP; Aitchison/CLR beta diversity; within-pair vs between-household distances; paired Wilcoxon genus tests on CLR with BH-FDR; explicit Prevotella check; fraction of reads lost to missing genus labels.

**Caveat:** n = 9 pairs; the smallest attainable paired Wilcoxon p-value is 2/512 = 0.0039. Results are exploratory.

In [1]:
!pip -q install scikit-bio statsmodels
import os, urllib.request
DATA='data'
os.makedirs(DATA,exist_ok=True)
BASE='https://raw.githubusercontent.com/omar-moustafa-code/als-gut-check/main/data'
for f in ['metadata.tsv','asv_table.tsv','taxonomy.tsv']:
    if not os.path.exists(f'{DATA}/{f}'): urllib.request.urlretrieve(f'{BASE}/{f}',f'{DATA}/{f}')

In [2]:
import itertools, numpy as np, pandas as pd
from scipy import stats
from skbio.diversity import alpha_diversity, beta_diversity
from statsmodels.stats.multitest import multipletests
D=DATA
md=pd.read_csv(f"{D}/metadata.tsv",sep="\t",index_col=0)
asv=pd.read_csv(f"{D}/asv_table.tsv",sep="\t",index_col=0)
tax=pd.read_csv(f"{D}/taxonomy.tsv",sep="\t",index_col=0)
print("asv",asv.shape)
depth=asv.sum(); md["depth"]=depth[md.index]
print(md.groupby("status").depth.describe()[["mean","min","50%","max"]])
pv=md.pivot(index="pair",columns="status",values="depth")
print("depth paired wilcoxon",stats.wilcoxon(pv.ALS,pv.Control))
ms=max(2,int(np.ceil(.05*asv.shape[1])))
a=asv[(asv>0).sum(axis=1)>=ms]; print("filtered",a.shape,"ms",ms, "reads kept frac",a.sum().sum()/asv.sum().sum())
rel=a.div(a.sum(axis=0),axis=1)
ids=list(md.index)

asv (275, 18)
                mean     min     50%     max
status                                      
ALS      8121.000000  6126.0  8294.0  9940.0
Control  8244.777778  6462.0  8192.0  9300.0
depth paired wilcoxon WilcoxonResult(statistic=np.float64(18.0), pvalue=np.float64(0.65234375))
filtered (121, 18) ms 2 reads kept frac 0.8484778535154659


## Alpha diversity (paired Wilcoxon)

In [3]:
# alpha
sh=alpha_diversity("shannon",a[ids].T.values,ids=ids)
rich=(a[ids]>0).sum()
pe=sh/np.log2(rich)
for n,v in [("shannon",sh),("richness",rich),("pielou",pe)]:
    t=pd.DataFrame({"v":v,"s":md.status,"p":md.pair}).pivot(index="p",columns="s",values="v")
    print(n,"ALS med",t.ALS.median(),"Ctl med",t.Control.median(),"diff",(t.ALS-t.Control).mean(),stats.wilcoxon(t.ALS,t.Control))

shannon ALS med 2.8382399288926727 Ctl med 2.2805961879955357 diff 0.5903582032309923 WilcoxonResult(statistic=np.float64(0.0), pvalue=np.float64(0.00390625))
richness ALS med 36.0 Ctl med 25.0 diff 11.444444444444445 WilcoxonResult(statistic=np.float64(0.0), pvalue=np.float64(0.00390625))
pielou ALS med 0.5575149517892942 Ctl med 0.5032194292009012 diff 0.059237698496066006 WilcoxonResult(statistic=np.float64(2.0), pvalue=np.float64(0.01171875))


## Rarefaction sensitivity check

In [4]:
# depth-sensitivity: rarefy to min depth
rng=np.random.default_rng(1); md_=int(a.sum().min())
def rar(c):
    p=np.repeat(np.arange(len(c)),c.values.astype(int)); s=rng.choice(p,md_,replace=False); return np.bincount(s,minlength=len(c))
R=pd.DataFrame({i:rar(a[i]) for i in ids},index=a.index)
shr=alpha_diversity("shannon",R.T.values,ids=ids)
t=pd.DataFrame({"v":shr,"s":md.status,"p":md.pair}).pivot(index="p",columns="s",values="v")
print("rarefied depth",md_,"shannon",stats.wilcoxon(t.ALS,t.Control))

rarefied depth 2277 shannon WilcoxonResult(statistic=np.float64(0.0), pvalue=np.float64(0.00390625))


## Beta diversity: Bray-Curtis, exact paired PERMANOVA, PERMDISP

In [5]:
# beta
def pseudoF(dm,grp):
    D2=dm**2;N=len(grp);g=np.unique(grp);SST=D2[np.triu_indices(N,1)].sum()/N
    SSW=0
    for k in g:
        ix=np.where(grp==k)[0];sub=D2[np.ix_(ix,ix)];SSW+=sub[np.triu_indices(len(ix),1)].sum()/len(ix)
    return ((SST-SSW)/(len(g)-1))/(SSW/(N-len(g)))
def paired_perm(dm,md):
    grp=(md.status=="ALS").values.astype(int); pairs=md.pair.values
    obs=pseudoF(dm,grp); up=sorted(set(pairs)); idx={p:np.where(pairs==p)[0] for p in up}
    cnt=0;tot=0
    for sw in itertools.product([0,1],repeat=len(up)):
        g=grp.copy()
        for s,p in zip(sw,up):
            if s: g[idx[p]]=1-g[idx[p]]
        tot+=1; cnt+= pseudoF(dm,g)>=obs-1e-12
    return obs,cnt/tot,tot
def unres(dm,md,n=9999):
    grp=(md.status=="ALS").values.astype(int);obs=pseudoF(dm,grp);r=np.random.default_rng(0);c=1
    for _ in range(n): c+=pseudoF(dm,r.permutation(grp))>=obs
    return obs,c/(n+1)
def disp(dm,md):
    # PERMDISP euclid-approx: distance to group centroid in PCoA space via Gower
    N=len(md);A=-.5*dm**2;J=np.eye(N)-1/N;G=J@A@J;w,v=np.linalg.eigh(G);pos=w>1e-10
    X=v[:,pos]*np.sqrt(w[pos]);grp=md.status.values;z=np.zeros(N)
    for k in np.unique(grp):
        ix=grp==k;z[ix]=np.linalg.norm(X[ix]-X[ix].mean(0),axis=1)
    zz=pd.Series(z,index=md.index);t=pd.DataFrame({"v":zz,"s":md.status,"p":md.pair}).pivot(index="p",columns="s",values="v")
    return stats.f_oneway(z[grp=="ALS"],z[grp=="Control"]),stats.wilcoxon(t.ALS,t.Control),z[grp=="ALS"].mean(),z[grp=="Control"].mean()
M=md.loc[ids]
bc=beta_diversity("braycurtis",rel[ids].T.values,ids=ids); B=bc.data
print("BC unrestricted",unres(B,M)); print("BC paired exact",paired_perm(B,M)); print("BC PERMDISP",disp(B,M))
bcr=beta_diversity("braycurtis",R[ids].T.values.astype(float),ids=ids).data
print("BC rarefied paired exact",paired_perm(bcr,M))

BC unrestricted (np.float64(1.7058967197501005), np.float64(0.0198))
BC paired exact (np.float64(1.7058967197501005), np.float64(0.0078125), 512)
BC PERMDISP (F_onewayResult(statistic=np.float64(0.8946420246249324), pvalue=np.float64(0.35828890253510615)), WilcoxonResult(statistic=np.float64(15.0), pvalue=np.float64(0.42578125)), np.float64(0.5243568999280948), np.float64(0.5585887696494942))
BC rarefied paired exact (np.float64(1.7057953126874632), np.float64(0.0078125), 512)


## Aitchison (CLR) distance

In [6]:
# CLR/Aitchison
x=a[ids].values.astype(float)+0.5; clr=np.log(x)-np.log(x).mean(0)
from scipy.spatial.distance import pdist,squareform
Ai=squareform(pdist(clr.T)); print("Aitchison unres",unres(Ai,M,999)); print("Aitchison paired exact",paired_perm(Ai,M)); print("Aitchison PERMDISP",disp(Ai,M))

Aitchison unres (np.float64(1.5888896084042725), np.float64(0.013))


Aitchison paired exact (np.float64(1.5888896084042725), np.float64(0.00390625), 512)
Aitchison PERMDISP (F_onewayResult(statistic=np.float64(8.18817853165963), pvalue=np.float64(0.011308547078116777)), WilcoxonResult(statistic=np.float64(2.0), pvalue=np.float64(0.01171875)), np.float64(23.84505495458949), np.float64(20.723120079681188))


## Within-pair vs between-household distances

In [7]:
# within pair vs between-household
pairs=M.pair.values;ia=[];ib=[]
for i in range(len(ids)):
    for j in range(i+1,len(ids)):
        (ia if pairs[i]==pairs[j] else ib).append(B[i,j])
print("within-pair BC mean",np.mean(ia),"between",np.mean(ib))

within-pair BC mean 0.7656935580445307 between 0.8327937575383134


## Genus level: paired Wilcoxon on CLR + BH-FDR (legacy Mann-Whitney for comparison)

In [8]:
# genus
g=tax.loc[a.index,"Genus"].fillna("Unclassified").replace("","Unclassified")
print("rel abundance with missing genus",rel[ids].loc[g=="Unclassified"].sum().mean())
rg=rel[ids].groupby(g).sum(); rg=rg.drop("Unclassified",errors="ignore")
pr=(rg>0).sum(axis=1); print("genera",len(rg))
rows=[]
cl=clr
gc=pd.DataFrame(a[ids].values.astype(float),index=a.index,columns=ids).groupby(g).sum().drop("Unclassified",errors="ignore")+0.5
gclr=np.log(gc)-np.log(gc).mean()
als=M.index[M.status=="ALS"];ctl=M.index[M.status=="Control"]
# pair order
ao=M[M.status=="ALS"].sort_values("pair").index; co=M[M.status=="Control"].sort_values("pair").index
for gn in rg.index:
    d=gclr.loc[gn,ao].values-gclr.loc[gn,co].values
    try: p=stats.wilcoxon(d).pvalue
    except: p=1
    mw=stats.mannwhitneyu(rg.loc[gn,ao],rg.loc[gn,co]).pvalue
    rows.append((gn,rg.loc[gn,ao].mean(),rg.loc[gn,co].mean(),d.mean(),p,mw,(rg.loc[gn]>0).sum(), (d>0).sum()))
T=pd.DataFrame(rows,columns=["genus","ALS_mean","Ctl_mean","meanCLRdiff","p_pairedWilcoxCLR","p_MWU_rel","n_present","pairs_ALS_higher"])
T["q_paired"]=multipletests(T.p_pairedWilcoxCLR,method="fdr_bh")[1];T["q_MWU"]=multipletests(T.p_MWU_rel,method="fdr_bh")[1]
T=T.sort_values("p_pairedWilcoxCLR");T.to_csv("genus_results.csv",index=False)
print(T.head(12).round(4).to_string())
print(T[T.genus.str.contains("Prevotella")].round(4).to_string())

rel abundance with missing genus 0.18737356133913646
genera 51


                           genus  ALS_mean  Ctl_mean  meanCLRdiff  p_pairedWilcoxCLR  p_MWU_rel  n_present  pairs_ALS_higher  q_paired   q_MWU
8                        Blautia    0.0218    0.0053       2.9031             0.0039     0.0111         10                 9    0.0664  0.1410
32                 Oscillibacter    0.0000    0.0006      -1.5745             0.0039     0.1693          2                 0    0.0664  0.3454
50  [Ruminococcus] torques group    0.0060    0.0003       2.3493             0.0039     0.0036          8                 9    0.0664  0.0927
26                 Lactobacillus    0.0000    0.0012      -1.6185             0.0078     0.1693          2                 1    0.0996  0.3454
20                 Fusobacterium    0.0041    0.0090      -0.9268             0.0195     1.0000          2                 2    0.1245  1.0000
33               Parabacteroides    0.0197    0.0004       3.4100             0.0195     0.0013          9                 7    0.1245  0.0683

## Phylum means (%)

In [9]:
# phylum
ph=tax.loc[a.index,"Phylum"].fillna("Unclassified");pm=rel[ids].groupby(ph).sum()
print((pm.T.groupby(M.status).mean().T*100).round(2).sort_values("ALS",ascending=False).head(6))


status               ALS  Control
Phylum                           
Bacteroidota       49.68    60.96
Firmicutes         39.72    27.69
Campilobacterota    4.25     4.32
Proteobacteria      2.73     1.32
Actinobacteriota    1.46     1.67
Verrucomicrobiota   1.23     2.93


## Figures

In [10]:
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
from skbio.stats.ordination import pcoa
COL={"ALS":"#2a6fbb","Control":"#c46a10"}; MK={"ALS":"o","Control":"s"}
plt.rcParams.update({"axes.spines.top":False,"axes.spines.right":False,"font.size":10})
ao=M[M.status=="ALS"].sort_values("pair").index; co=M[M.status=="Control"].sort_values("pair").index
fig,axs=plt.subplots(1,3,figsize=(13,4))
for ax,(name,v) in zip(axs,[("Shannon (bits)",sh),("Richness (ASVs)",rich),("Pielou evenness",pe)]):
    for a_,c_ in zip(ao,co): ax.plot([0,1],[v[c_],v[a_]],color="#999",lw=1,zorder=1)
    ax.scatter(np.zeros(9),v[co],c=COL["Control"],marker="s",s=40,zorder=2); ax.scatter(np.ones(9),v[ao],c=COL["ALS"],s=40,zorder=2)
    ax.set_xticks([0,1]); ax.set_xticklabels(["Control","ALS"]); ax.set_xlim(-.3,1.3); ax.set_ylabel(name)
fig.suptitle("Alpha diversity, paired by household (lines join spouses)"); fig.tight_layout(); fig.savefig("fig1_alpha_paired.png",dpi=200); plt.show()

In [11]:
ord_=pcoa(bc,number_of_dimensions=3); P=ord_.samples; pe_=ord_.proportion_explained
fig,ax=plt.subplots(figsize=(5.5,5))
for p_ in sorted(set(M.pair)):
    i=[x for x in ids if M.pair[x]==p_]; ax.plot(P.loc[i,"PC1"],P.loc[i,"PC2"],color="#999",lw=1,zorder=1)
for s in ["ALS","Control"]:
    i=M.index[M.status==s]; ax.scatter(P.loc[i,"PC1"],P.loc[i,"PC2"],c=COL[s],marker=MK[s],s=60,label=s,zorder=2)
ax.set_xlabel(f"PC1 ({pe_['PC1']*100:.1f}%)"); ax.set_ylabel(f"PC2 ({pe_['PC2']*100:.1f}%)"); ax.legend(frameon=False)
ax.set_title("Bray-Curtis PCoA (lines join spouses)"); fig.tight_layout(); fig.savefig("fig2_pcoa.png",dpi=200); plt.show()

In [12]:
# rarefaction curves + within-pair vs between-household distances
fig,axs=plt.subplots(1,2,figsize=(11,4)); depths=np.linspace(100,int(a.sum().min()),12,dtype=int); rr=np.random.default_rng(2)
for i in ids:
    y=[]
    for d_ in depths:
        p=np.repeat(np.arange(a.shape[0]),a[i].values.astype(int)); s=rr.choice(p,d_,replace=False); y.append((np.bincount(s,minlength=a.shape[0])>0).sum())
    axs[0].plot(depths,y,color=COL[M.status[i]],lw=1)
axs[0].set_xlabel("Reads sampled"); axs[0].set_ylabel("Observed ASVs"); axs[0].set_title("Rarefaction (blue = ALS, orange = Control)")
axs[1].boxplot([ia,ib],widths=.5); axs[1].set_xticks([1,2]); axs[1].set_xticklabels(["Within pair","Between households"]); 
for k,v in enumerate([ia,ib],1): axs[1].scatter(np.random.default_rng(3).normal(k,.05,len(v)),v,s=12,color="#555",alpha=.6)
axs[1].set_ylabel("Bray-Curtis dissimilarity"); axs[1].set_title("Spouses are more similar than strangers")
fig.tight_layout(); fig.savefig("fig3_rarefaction_household.png",dpi=200); plt.show()

## Robust genus analysis
Prevalence filter (>=4 samples), three pseudocounts, household random-intercept linear mixed model on CLR, and bootstrap CIs on the paired difference.

In [13]:
# Robust genus analysis: stricter prevalence filter, pseudocount sensitivity, household random-intercept model, bootstrap CIs
import statsmodels.formula.api as smf, warnings; warnings.filterwarnings("ignore")
gcnt=pd.DataFrame(a[ids].values.astype(float),index=a.index,columns=ids).groupby(g).sum().drop("Unclassified",errors="ignore")
prev=(gcnt>0).sum(axis=1); keep=prev[prev>=4].index   # present in >=4 of 18 samples (>=~22%)
print("genera tested after prevalence filter:",len(keep),"of",len(gcnt))
def clr_(pc):
    x=gcnt.loc[keep]+pc; return np.log(x)-np.log(x).mean()
res=[]; rb=np.random.default_rng(0)
for pc in [0.1,0.5,1.0]:
    cl=clr_(pc)
    for gn in keep:
        d=cl.loc[gn,ao].values-cl.loc[gn,co].values
        pw=stats.wilcoxon(d).pvalue if np.any(d!=0) else 1.0
        res.append((pc,gn,d.mean(),pw))
R_=pd.DataFrame(res,columns=["pseudocount","genus","meanCLRdiff","p"])
R_["q"]=R_.groupby("pseudocount").p.transform(lambda p: multipletests(p,method="fdr_bh")[1])
piv=R_.pivot(index="genus",columns="pseudocount",values="q"); print("min q by pseudocount:\n",piv.min().round(3))
# household random-intercept model on CLR(0.5)
cl=clr_(0.5); rows=[]
df0=M.copy(); df0["ALS"]=(df0.status=="ALS").astype(int)
for gn in keep:
    df0["y"]=cl.loc[gn,df0.index].values
    try:
        m=smf.mixedlm("y ~ ALS",df0,groups=df0["pair"]).fit(reml=True); b,se,p=m.params["ALS"],m.bse["ALS"],m.pvalues["ALS"]
    except Exception: b=se=p=np.nan
    d=cl.loc[gn,ao].values-cl.loc[gn,co].values; bs=[rb.choice(d,9).mean() for _ in range(2000)]
    signs=np.array(list(itertools.product([-1,1],repeat=9))); perm_p=(np.abs((signs*d).mean(axis=1))>=abs(d.mean())-1e-12).mean()  # exact sign-flip test, 512 flips
    rows.append((gn,prev[gn],b,se,p,d.mean(),*np.percentile(bs,[2.5,97.5]),perm_p))
RM=pd.DataFrame(rows,columns=["genus","n_present","LMM_beta","LMM_se","LMM_p","meanCLRdiff","boot_lo","boot_hi","exact_perm_p"]).dropna(subset=["LMM_p"])
RM["LMM_q"]=multipletests(RM.LMM_p,method="fdr_bh")[1]; RM["exact_perm_q"]=multipletests(RM.exact_perm_p,method="fdr_bh")[1]; RM=RM.sort_values("LMM_p"); RM.to_csv("genus_robust_results.csv",index=False)
print(RM.head(10).round(3).to_string())

genera tested after prevalence filter: 32 of 51
min q by pseudocount:
 pseudocount
0.1    0.062
0.5    0.062
1.0    0.062
dtype: float64


                           genus  n_present  LMM_beta  LMM_se  LMM_p  meanCLRdiff  boot_lo  boot_hi  exact_perm_p  LMM_q  exact_perm_q
5                        Blautia         10     2.632   0.564  0.000        2.632    1.603    3.655         0.004  0.000         0.062
31  [Ruminococcus] torques group          8     2.078   0.454  0.000        2.078    1.212    2.851         0.004  0.000         0.062
10          Escherichia-Shigella          7    -3.797   0.970  0.000       -3.797   -5.431   -1.701         0.016  0.001         0.125
19               Parabacteroides          9     3.139   0.910  0.001        3.139    1.389    4.722         0.020  0.005         0.125
27                    Tuzzerella          5    -2.540   0.831  0.002       -2.540   -4.010   -0.617         0.035  0.014         0.179
8                      Dialister         13    -3.228   1.105  0.003       -3.228   -5.276   -1.211         0.020  0.019         0.125
26               Subdoligranulum         10     2.103  

In [14]:
top=RM.sort_values("meanCLRdiff"); top=pd.concat([top.head(8),top.tail(8)])
fig,ax=plt.subplots(figsize=(6.5,5.5)); y=np.arange(len(top))
ax.errorbar(top.meanCLRdiff,y,xerr=[top.meanCLRdiff-top.boot_lo,top.boot_hi-top.meanCLRdiff],fmt="o",color="#2a6fbb",ecolor="#888",capsize=2)
ax.axvline(0,color="#444",lw=1); ax.set_yticks(y); ax.set_yticklabels(top.genus); ax.set_xlabel("Mean paired CLR difference (ALS - Control), 95% bootstrap CI")
ax.set_title("Genus effect sizes (exploratory)"); fig.tight_layout(); fig.savefig("fig4_genus_effects.png",dpi=200); plt.show()

## Interpretation notes
- Depth is balanced between groups, and the Shannon result survives rarefaction.
- Bray-Curtis PERMDISP is non-significant, so the PERMANOVA signal is a location shift; Aitchison dispersion differs between groups, so read the CLR PERMANOVA with that caveat.
- Many genera occur in only 2–3 samples; CLR with a pseudocount makes their p-values unreliable. Genus results are hypothesis-generating; none passes q<0.05.
- ~19% of filtered reads lack a genus label.
- Sex, diet, dysphagia and riluzole are unmodelled confounders.